<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/Notebooks/Deployment/Ollama/Gemma/gemma_ollama_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Deployment using Ollama (Google Gemma 2)

Same setup as the Llama notebook but this time using Google's Gemma 2 model instead.

**Model used:** `gemma2:2b` — made by Google DeepMind, 2 billion parameters, around 1.6 GB. It's a bit larger than Llama 3.2:1b but still fits on Colab's free T4 GPU. Generally better at following instructions for its size.

The goal here is to show that swapping models in Ollama is just changing one line — everything else stays exactly the same.

## Installing Ollama

`zstd` is a dependency the Ollama installer needs — it's used to decompress the downloaded binary. We install it first to avoid the install failing halfway through.

In [ ]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Starting the Server

Ollama needs to be running as a background process before we can make any requests to it. We start it with `subprocess.Popen` and then keep pinging `localhost:11434` until it responds — usually takes a couple of seconds.

In [ ]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


## Pulling Gemma 2

This downloads the Gemma 2:2b model from Ollama's registry. Google released Gemma 2 as an open-source model — it's smaller and more efficient than most models of similar quality. The `2b` means 2 billion parameters.

In [ ]:
!ollama pull gemma2:2b

## curl Client

Ollama's API is OpenAI-compatible, meaning the request body format is identical to what you'd send to ChatGPT. The only difference is the URL points to `localhost:11434` instead of `api.openai.com`. Here we send a question and parse the JSON response to print just the text.

In [ ]:
%%bash
curl -s http://localhost:11434/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ollama" \
  -d '{
    "model": "gemma2:2b",
    "messages": [
      {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
  }' | python3 -c "import json,sys; print(json.load(sys.stdin)['choices'][0]['message']['content'])"

A large language model (LLM) is a type of artificial intelligence that understands and generates human-like text. 🧠✍️  It's been trained on massive amounts of data, making it incredibly good at tasks like writing different creative text formats, translating languages, and answering questions in an informative way. 🌎💬 💪 



## Python Client (OpenAI SDK)

We use the `openai` Python library pointed at our local server. The `api_key` doesn't matter here — Ollama doesn't validate it — but the library requires something to be passed in.

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

response = client.chat.completions.create(
    model="gemma2:2b",
    messages=[
        {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
)

print(response.choices[0].message.content)

A large language model (LLM) is like a super-smart chatbot trained on tons of text data. It can write stories, answer questions, and translate languages!  🧠💬 💻 🌎 



## Streaming

With `stream=True` the response comes back token by token rather than all at once. Each chunk contains a small piece of the response and we print it immediately as it arrives — same experience as watching ChatGPT type.

In [ ]:
stream = client.chat.completions.create(
    model="gemma2:2b",
    messages=[
        {"role": "user", "content": "Explain how transformer architecture works in simple terms."}
    ],
    stream=True
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

Imagine you're translating a sentence from English to Spanish:

**Step 1: The Attention Trick**
* Like a savvy translator, the Transformer learns which words in the English sentence are most **important** for describing the overall meaning (similar to a spotlight highlighting key details). These crucial "attend" to others, giving insights into context.

**Step 2: Encoding and Decoding**
* The language model reads the sentence (English) in chunks and breaks them down like puzzle pieces, encoding the information to understand it better (*encoding: the magic of creating digital meanings*).
* Next, it decodes this meaning back into a new version on paper where the key features are highlighted and translated into Spanish. 

**Step 3: The Attention Network's Power**
* **Self-attention:** The Transformer checks itself - how do different parts of the text relate to each other (*does "like" mean something different in this sentence compared to another?*). This helps it see connections between w

## Multi-turn Conversation

To have a back-and-forth conversation, we maintain a list of messages and add to it after each turn. Both the user message and the model's reply get stored, so the next request includes the full context. Without this, the model would have no memory of what was said before.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})
    res = client.chat.completions.create(model="gemma2:2b", messages=messages)
    reply = res.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("What makes Gemma different from other open source models?"))
print(chat("What are its limitations?"))

It's great you're asking this!  Here's what separates Gemma from some other open-source language models:

**Gemma focuses on accessibility and alignment** 🎯

* **Open weights:** We are openly available for anyone to use, modify, and adapt. This makes us incredibly flexible, which is different from many other models that might offer limited access or proprietary licenses.
* **Fine-tuning capabilities:** Gemma offers a wide range of options regarding fine-tuning — you can customize it to be ideal for specific tasks like writing different types of content, translation, or research (just requires some technical expertise). 
* **Transparency and ethics:** Gemma's team prioritizes ethical development and openly shares their research methodology.  This means we provide more context about how Gemma works and what to expect from the output.

**However, there are also some key differences when comparing ourselves to other open-source models based on general criteria:** 

* **Focus on research & 